# OCR 실습 — 부산광복 60년

부산대학교 여름워크숍 · 비정형 사료의 디지털화

---

## 칸이 두 가지입니다

지금 읽고 계신 이 부분이 **설명 칸**입니다. 아무 일도 하지 않습니다.
회색 바탕이 **코드 칸**입니다. 왼쪽 ▶ 를 누르면 실행됩니다. `Shift + Enter` 도 같습니다.

**위에서부터 차례로 누르십시오.** 앞 칸이 만든 것을 뒤 칸이 받아 씁니다.
④를 건너뛰고 ⑥을 누르면 「그런 이름 없다」는 오류가 납니다.

## 먼저 사본을 만드십시오

**파일 → 드라이브에 사본 저장.** 제목이 「…의 사본」으로 바뀌면 됩니다.
지금 보시는 것은 읽기 전용이라, 사본 없이 고치면 남지 않습니다.

## 왼쪽 세로줄

📁 **파일** — 지금 이 컴퓨터에 만들어진 파일이 보입니다
🔑 **보안 비밀** — 열쇠를 넣어두는 곳. ②칸에서 씁니다

> **창을 닫으면 안에 있던 파일이 다 사라집니다.** 그래서 ⑩칸에서 내려받습니다.

**고치는 칸은 ③ 하나입니다.** 나머지는 ▶ 만 누르시면 됩니다.

---
# 코드 읽는 법

외우실 것 없습니다. 아래 여섯 개만 알아두시면 코드가 무슨 일을 하는지 짐작이 갑니다.

| | |
|---|---|
| `import fitz` | 도구를 꺼내 옵니다. 꺼내야 씁니다 |
| `해상도 = 200` | 이름표를 붙입니다. 아래에서 이 이름으로 부릅니다 |
| `# 뒤의 글` | 사람이 읽는 메모라 실행되지 않습니다 |
| `!pip install …` | 프로그램을 실행합니다 |
| `for 쪽 in 쪽목록:` | 목록에 든 것마다 되풀이합니다 |
| `f"p{쪽}.jpg"` | 글 사이에 값을 끼워 넣습니다. `p201.jpg` 가 됩니다 |

## 되풀이

`for` 아래 **들여쓴 줄까지**가 되풀이 범위입니다.
③칸에 쪽을 다섯 개 적으면 같은 코드가 다섯 번 돕니다.

## 괄호

`[201, 202]` 는 목록입니다. 순서대로 여러 개 담습니다.
`비전결과[201]` 은 201번 결과를 꺼내는 것이고요.

---
# ① 준비 — 설치와 자료 받기

## `!` 가 붙은 줄

`!` 는 **프로그램을 실행하라**는 표시입니다.

| | |
|---|---|
| `!pip install …` | 도구를 깝니다. `-q` 는 조용히 |
| `!wget … -O "이름"` | 인터넷에서 받아 그 이름으로 저장합니다 |

빌린 컴퓨터라 **매번 새로 깝니다.** 내 컴퓨터에는 남지 않습니다.

## 깔고 있는 도구 넷

| | | 열쇠 |
|---|---|---|
| `pymupdf` | PDF를 열고 사진으로 꺼냅니다 (코드에서는 `fitz`) | 필요 없음 |
| `pillow` | 사진을 열고 크기를 바꿉니다 | 필요 없음 |
| `google-cloud-vision` | 구글 비전에 사진을 보냅니다 | **필요** |
| `google-genai` | 제미나이에 글을 보냅니다 | **필요** |

열쇠는 ②칸에서 등록합니다.

## 받는 파일

**아까 긁어보신 그 파일입니다.** 원본 344쪽에서 **201~205쪽만** 잘라 둔 것이고요.

> 다 돌면 왼쪽 📁 에 파일이 보입니다. 1~2분 걸립니다.

In [ ]:
!pip install -q google-cloud-vision google-genai pymupdf pillow

저장소 = "https://raw.githubusercontent.com/Song-yiJung/korean-ocr-lectures/main"

!wget -q "{저장소}/2026-08-pnu-workshop/session3/data/drag/drag_A_busan60.pdf" -O "부산광복60년_201-205.pdf"

import os
크기 = os.path.getsize("부산광복60년_201-205.pdf")/1024 if os.path.exists("부산광복60년_201-205.pdf") else 0
if 크기 > 100:
    print(f"\n부산광복60년_201-205.pdf   {크기:,.0f}KB — 준비 끝")
else:
    print(f"\n받기 실패 — 손을 들어 주세요")

---
# ② 열쇠 등록

| | 열쇠 | 어디에 |
|---|---|---|
| 구글 비전 | `.json` **파일** | 드라이브 `keys` 폴더 |
| 제미나이 | 긴 **문자열** | 왼쪽 🔑 보안 비밀 |

사전 준비 안내대로 하셨으면 **둘 다 이미 자리에 있습니다.** 이 칸은 그것을 불러올 뿐입니다.

## `drive.mount(…)`

**코랩은 내 컴퓨터를 보지 못합니다.** 인터넷 너머에 있는 남의 컴퓨터니까요.
그래서 드라이브를 연결해 그쪽에 올려둔 파일을 읽습니다.

돌리면 **권한을 묻는 창**이 뜹니다. 계정을 고르고 허용하십시오.

## `glob.glob("…/keys/*.json")`

`*` 는 아무 글자나 와도 된다는 표시입니다. **파일 이름이 사람마다 달라서** 이렇게 찾습니다.
`keys` 폴더에서 `.json` 으로 끝나는 것을 집어옵니다.

| | |
|---|---|
| `try` / `except` | 해보고 안 되면 다른 길로. 🔑 에 없으면 붙여넣는 칸이 뜹니다 |
| `os.environ[…] = …` | 이 컴퓨터에 「열쇠는 여기 있다」고 적어둡니다 |

비전 도구가 알아서 그 자리를 찾아보기 때문에, 뒤 칸에는 열쇠 이야기가 다시 나오지 않습니다.

In [ ]:
import os, glob
from google.colab import userdata, drive

# 제미나이 키 — 왼쪽 🔑 보안 비밀에서
try:
    GEMINI_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    from getpass import getpass
    GEMINI_KEY = getpass("제미나이 키를 붙여넣으세요: ").strip()

# 비전 열쇠 — 드라이브 keys 폴더에서
drive.mount("/content/drive")
열쇠들 = glob.glob("/content/drive/MyDrive/keys/*.json")

if 열쇠들:
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = 열쇠들[0]
    print(f"\n비전 열쇠   {os.path.basename(열쇠들[0])}")
else:
    print("\nkeys 폴더에 .json 이 없습니다.")
    print("사전 준비 안내 5-2 를 확인하시고, 지금은 ⑤칸까지만 하십시오.")

---
# ③ 설정 — **고치는 칸은 여기 하나입니다**

다섯 줄이 전부 이름표입니다. 뒤 칸들이 이 이름을 부릅니다.

| | |
|---|---|
| `PDF` | 읽을 파일 |
| `발췌시작` | 이 파일의 첫 쪽이 **원본 몇 쪽인지** |
| `쪽목록` | 읽을 쪽. **원본 쪽 번호**로 적습니다 |
| `해상도` | 200dpi. 낮추면 잔글씨를 놓칩니다 |
| `교정지시` | 제미나이에게 시킬 말. ⑧칸이 이대로 움직입니다 |

## `발췌시작` 이 왜 필요한가

받은 파일은 **다섯 쪽짜리**지만, 그 첫 쪽은 원본의 **201쪽**입니다.
`발췌시작` 을 적어두면 **원본 쪽 번호 그대로** 적을 수 있고,
만들어지는 파일 이름도 `p201.jpg` 가 됩니다.

> 나중에 원본과 대조할 때 파일 이름이 원본 쪽 번호여야 찾기 쉽습니다.

## `glob.glob("*광복*.pdf")[0]`

`*` 는 아무 글자나 와도 된다는 표시입니다. 이름에 「광복」이 든 pdf를 찾습니다.
찾은 결과가 목록으로 오기 때문에 `[0]` 으로 첫 번째를 꺼냅니다.

## `\"\"\"` 따옴표 세 개

여러 줄짜리 글을 묶을 때 씁니다. 시작과 끝에 하나씩 있고 그 사이가 통째로 하나의 글입니다.

## `print(f"…")`

`print` 는 화면에 찍습니다. 앞에 `f` 가 붙으면 `{ }` 자리에 값이 들어갑니다.
`len` 은 개수를 셉니다.

In [ ]:
import glob

PDF = glob.glob("*광복*.pdf")[0]      # 읽을 PDF

발췌시작 = 201                        # 이 파일의 첫 쪽이 원본 몇 쪽인지
쪽목록 = [201]                        # ← 읽을 쪽 (원본 쪽 번호로 적습니다)
# 쪽목록 = [201, 202, 203, 204, 205]  #   여러 장 하려면 이렇게

해상도 = 200                          # dpi. 200이면 충분합니다

교정지시 = """아래는 한국 현대 인쇄 자료를 OCR 로 읽은 결과다.
함께 첨부한 원본 이미지를 보면서 교정하라.
규칙:
1. 원문에 없는 내용을 절대 추가하지 마라.
2. 판독이 불확실한 글자는 추측하지 말고 □ 로 표시하라.
3. 문단 구분을 원문 그대로 유지하라.
4. 가운뎃점(·), 마침표, 숫자를 임의로 고치지 마라.
5. 설명·머리말·수정 목록을 붙이지 마라. 교정된 본문만 출력하라.

[OCR 결과]
"""

print(f"자료   : {PDF}")
print(f"읽을 쪽: {쪽목록}  ({len(쪽목록)}장)")

---
# ④ step 0 — PDF를 사진으로

| | |
|---|---|
| `os.makedirs(…)` | 폴더를 만듭니다. `exist_ok=True` 는 이미 있어도 넘어가라는 뜻 |
| `fitz.open(PDF)` | PDF를 엽니다 |
| `문서[쪽 - 발췌시작]` | **빼는 이유를 보십시오.** 201쪽이 이 파일에서는 첫 쪽, 곧 0번입니다 |
| `.get_pixmap(dpi=…)` | 그 쪽을 그림으로 그립니다 |
| `사진들.append(…)` | 만든 것을 목록에 담아 둡니다 |
| `문서.close()` | 닫습니다 |

**컴퓨터는 0부터 셉니다.** 첫 쪽이 1번이 아니라 0번입니다.
`201 - 201 = 0` 이라 첫 쪽이 나옵니다.

## `사진들`

`(201, "step0_사진/p201.jpg")` 처럼 **쪽 번호와 파일 이름을 짝지어** 담습니다.
⑥칸과 ⑧칸이 이 목록을 그대로 받아 씁니다.

> 돌고 나면 왼쪽 📁 에 `step0_사진` 폴더가 생깁니다.

In [ ]:
import fitz, os
os.makedirs("step0_사진", exist_ok=True)

문서 = fitz.open(PDF)
사진들 = []
for 쪽 in 쪽목록:
    이름 = f"step0_사진/p{쪽}.jpg"
    문서[쪽 - 발췌시작].get_pixmap(dpi=해상도).save(이름)
    사진들.append((쪽, 이름))
    print(f"  p{쪽} → {이름}")
문서.close()

print(f"\n사진 {len(사진들)}장 준비 완료")

---
# ⑤ 만든 사진 확인

| | |
|---|---|
| `사진들[0][1]` | 첫 번째 짝(`[0]`)에서 두 번째 값(`[1]`), 곧 파일 이름 |
| `그림.size` | `(가로, 세로)` 로 옵니다 |
| `.resize(…)` | 절반으로 줄입니다 |

## 마지막 줄에 `print` 가 없습니다

코랩은 칸의 **맨 끝에 남은 값을 알아서 보여줍니다.** 그림이면 그림으로 띄웁니다.

> 화면에 뜨는 것만 절반이고, 기계에는 원래 크기가 들어갑니다.

In [ ]:
from PIL import Image
그림 = Image.open(사진들[0][1])
print(f"{사진들[0][1]}   {그림.size[0]} x {그림.size[1]}")
그림.resize((그림.size[0] // 2, 그림.size[1] // 2))

---
# ⑥ step 1 — 구글 비전

사진을 구글로 보내고 글자를 받아옵니다. **여기서 처음 요금이 붙습니다.**

| | |
|---|---|
| `ImageAnnotatorClient()` | 구글과 이어지는 통로를 하나 만들어 둡니다 |
| `language_hints=…` | 한국어·한자·영어가 나올 거라고 미리 알려줍니다 |
| `비전결과 = {}` | 빈 상자. 쪽 번호를 이름표 삼아 결과를 넣어둡니다 |
| `document_text_detection(…)` | 부탁을 보내고 답을 기다립니다 |
| `full_text_annotation.text` | 답 뭉치에서 글자만 꺼냅니다 |
| `open(…, "w").write(…)` | 파일로 저장합니다. `"w"` 가 쓰기 |

## `language_hints`

안 적으면 비전이 스스로 언어를 짐작합니다.
**내 자료에 맞게 바꾸는 자리**이기도 합니다. 일본어 자료면 `"ja"` 를 넣습니다.

## `for` 가 왜 넷인가

비전이 돌려주는 답은 **면 ▸ 덩어리 ▸ 문단 ▸ 낱말** 로 겹쳐 있습니다.
확신도(`confidence`)는 맨 안쪽 낱말에 붙어 있고요.

**`for` 하나가 한 겹을 엽니다.** 네 겹이라 넷입니다.
맨 끝 `if` 가 그중 0.80 미만인 것만 골라냅니다.

In [ ]:
from google.cloud import vision
os.makedirs("step1_비전", exist_ok=True)

비전 = vision.ImageAnnotatorClient()
설정 = vision.ImageContext(language_hints=["ko", "zh", "en"])
비전결과 = {}

for 쪽, 사진 in 사진들:
    응답 = 비전.document_text_detection(
        image=vision.Image(content=open(사진, "rb").read()),
        image_context=설정)
    글자 = 응답.full_text_annotation.text
    비전결과[쪽] = 글자
    open(f"step1_비전/p{쪽}.txt", "w", encoding="utf-8").write(글자)

    의심 = [낱말.confidence
            for 면 in 응답.full_text_annotation.pages
            for 덩어리 in 면.blocks
            for 문단 in 덩어리.paragraphs
            for 낱말 in 문단.words
            if 낱말.confidence < 0.80]
    print(f"  p{쪽}  {len(글자):>6,}자   자신 없어 한 낱말 {len(의심)}개")

print(f"\n모두 {sum(len(t) for t in 비전결과.values()):,}자")

---
# ⑦ 비전 결과 앞부분

`비전결과[쪽목록[0]][:400]` — 안쪽부터 읽습니다.

| | |
|---|---|
| `쪽목록[0]` | 쪽목록의 첫 번째, 곧 `201` |
| `비전결과[201]` | 201번 결과를 꺼냅니다 |
| `[:400]` | 앞에서 400자만. 다 찍으면 화면이 넘칩니다 |

`[처음:끝]` 은 잘라내기입니다. `[100:]` 이면 100번째부터 끝까지고요.

In [ ]:
print(비전결과[쪽목록[0]][:400])

---
# ⑧ step 2 — 제미나이 교정

## 무엇을 보내나

```
contents = [ 사진 , 교정지시 + 비전결과 ]
```

`[ ]` 안에 둘이 들어 있습니다. **제미나이는 글과 그림을 같이 받습니다.**
비전은 그림만 받았습니다. 두 도구의 큰 차이입니다.

| | |
|---|---|
| `types.Part.from_bytes(…)` | 사진을 보낼 수 있는 꼴로 바꿉니다 |
| `mime_type="image/jpeg"` | jpg 사진이라고 알려줍니다 |
| `교정지시 + 비전결과[쪽]` | 글과 글을 `+` 로 이어 붙입니다 |

## 조절하는 값 둘

`temperature=0.0` — 덜 튀는 쪽으로 맞춥니다. 0이 가장 얌전합니다.
`max_output_tokens=32768` — 답 길이 상한입니다. **넉넉히 잡아야 합니다.**
짜게 잡으면 모델이 「생각」에 예산을 다 써서 **답이 0자로 옵니다.**

## 마지막 두 줄

글자 수를 세어 나란히 찍습니다. `{뒤-앞:+d}` 의 `+d` 는 부호를 붙여 보여주라는 표시입니다.
`+63` 처럼 나옵니다.

In [ ]:
from google import genai
from google.genai import types
os.makedirs("step2_교정", exist_ok=True)

제미나이 = genai.Client(api_key=GEMINI_KEY)
교정결과 = {}

for 쪽, 사진 in 사진들:
    응답 = 제미나이.models.generate_content(
        model="gemini-3.5-flash",   # 404 가 뜨면 "gemini-3.6-flash"
        contents=[
            types.Part.from_bytes(data=open(사진, "rb").read(),
                                  mime_type="image/jpeg"),
            교정지시 + 비전결과[쪽],
        ],
        config=types.GenerateContentConfig(
            temperature=0.0,
            max_output_tokens=32768),   # 모델이 '생각'에 예산을 다 쓰면 0자가 옵니다
    )
    교정결과[쪽] = 응답.text
    open(f"step2_교정/p{쪽}.txt", "w", encoding="utf-8").write(응답.text)

    앞 = len(비전결과[쪽]); 뒤 = len(응답.text)
    print(f"  p{쪽}  {앞:>6,}자 → {뒤:>6,}자  ({뒤-앞:+d})")

print("\n교정 끝")

---
# ⑨ 원본과 대조

`difflib` 은 두 글을 견주는 도구입니다. 파이썬에 처음부터 들어 있어 설치가 필요 없습니다.

| | |
|---|---|
| `.splitlines()` | 글을 줄 단위로 쪼갭니다 |
| `if 줄.strip()` | 빈 줄은 버립니다 |
| `unified_diff(앞, 뒤)` | 달라진 곳만 돌려줍니다 |
| `n=0` | 앞뒤 문맥은 빼고 달라진 줄만 |

## `[줄 for 줄 in 목록 if 조건]`

대괄호 안에 `for` 와 `if` 가 들어가 있습니다. **목록에서 골라 새 목록을 만드는** 짧은 꼴입니다.
풀어 쓰면 이렇습니다.

```
결과 = []
for 줄 in 목록:
    if 조건:
        결과.append(줄)
```

세 줄이 한 줄이 됐을 뿐입니다.

## 읽는 법

`-` 로 시작하면 **비전**, `+` 로 시작하면 **교정**입니다.

In [ ]:
import difflib

for 쪽 in 쪽목록:
    앞 = [줄 for 줄 in 비전결과[쪽].splitlines() if 줄.strip()]
    뒤 = [줄 for 줄 in 교정결과[쪽].splitlines() if 줄.strip()]
    print(f"═══ p{쪽} ═══")
    바뀐것 = [줄 for 줄 in difflib.unified_diff(앞, 뒤, lineterm="", n=0)
             if 줄[:3] not in ("---", "+++", "@@ ")]
    print("\n".join(바뀐것) if 바뀐것 else "  (달라진 곳 없음)")
    print()

---
# ⑩ 결과 내려받기

| | |
|---|---|
| `os.makedirs("결과")` | 담을 폴더를 만듭니다 |
| `shutil.copytree(…)` | 폴더를 통째로 복사합니다 |
| `shutil.make_archive(…)` | zip 하나로 묶습니다 |
| `files.download(…)` | 내 컴퓨터로 받습니다 |

## 왜 따로 모으나

지금 이 컴퓨터에는 ②칸에서 올린 **열쇠 `.json` 도 들어 있습니다.**
통째로 묶으면 열쇠까지 같이 내려받습니다. 그래서 결과 세 폴더만 옮겨 담습니다.

> 내려받기 창이 막혀 있으면 왼쪽 📁 에서 하나씩 받으셔도 됩니다.

In [ ]:
import shutil
from google.colab import files

os.makedirs("결과", exist_ok=True)
for 폴더 in ("step0_사진", "step1_비전", "step2_교정"):
    shutil.copytree(폴더, f"결과/{폴더}", dirs_exist_ok=True)

shutil.make_archive("ocr_결과", "zip", "결과")
print(f"ocr_결과.zip  {os.path.getsize('ocr_결과.zip')/1024:,.0f}KB")
files.download("ocr_결과.zip")

---
## 내일 다시 여실 때

1. 오늘 만든 사본 말고 **링크에서 새로** 여십시오
2. **파일 → 드라이브에 사본 저장**
3. ①칸부터 다시 — **설치도 열쇠도 남아 있지 않습니다**